In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from mnist import get_mnist_loaders
from LeNet import LeNet


# ------------------------
# Training / Eval
# ------------------------
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            preds = logits.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

def print_model_params(model):
    total = 0

    for name, param in model.named_parameters():
        if param.requires_grad:
            n = param.numel()
            total += n

            shape = str(tuple(param.shape))  # <-- fix

            print(f"{name:20s} | shape: {shape:15s} | params: {n}")

    print("-" * 60)
    print(f"Total trainable params: {total}")
# ------------------------
# Main
# ------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Import loaders
    train_loader, test_loader = get_mnist_loaders()

    model = LeNet(activation="tanh").to(device)
    print_model_params(model)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    epochs = 20

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion, device)
        train_acc = evaluate(model,train_loader,device)
        acc = evaluate(model, test_loader, device)

        print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {acc:.4f}")


if __name__ == "__main__":
    main()


Using device: cuda
conv1.weight         | shape: (6, 1, 5, 5)    | params: 150
conv1.bias           | shape: (6,)            | params: 6
conv2.weight         | shape: (16, 6, 5, 5)   | params: 2400
conv2.bias           | shape: (16,)           | params: 16
fc1.weight           | shape: (120, 256)      | params: 30720
fc1.bias             | shape: (120,)          | params: 120
fc2.weight           | shape: (84, 120)       | params: 10080
fc2.bias             | shape: (84,)           | params: 84
fc3.weight           | shape: (10, 84)        | params: 840
fc3.bias             | shape: (10,)           | params: 10
------------------------------------------------------------
Total trainable params: 44426
Epoch 01 | Loss: 0.2960 | Train Acc: 0.9767 | Test Acc: 0.9786
Epoch 02 | Loss: 0.0675 | Train Acc: 0.9858 | Test Acc: 0.9844
Epoch 03 | Loss: 0.0462 | Train Acc: 0.9901 | Test Acc: 0.9861
Epoch 04 | Loss: 0.0351 | Train Acc: 0.9917 | Test Acc: 0.9860
Epoch 05 | Loss: 0.0274 | Train Acc: 0

In [6]:
import symo.optim2 as symo_optim
from symo.group import S, B, I, O
from symo.factory2 import GroupsSpec, groups_spec